# Build to Verify
## Four-Bar Linkage Design for the MiniClaw Jaw (One Side)
## ME 493B — AI in Product Development | Mini-Project 4, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Friday, May 29, 2026 at 11:59 PM
**Time estimate:** 3–4 hours of focused work
**Points:** 50 (Part A). Part B is worth 50 points separately and is
your team's integration of the individual four-bar designs into one
prototype-ready MiniClaw.

---

### What changed

MP1 was *define the problem.* MP2 was *research the problem.* MP3 was
*productize your engineering work.* MP4 Part A is **use the productized
stack to design a real subsystem and verify it holds up**, before team
integration in Part B.

You arrive at MP4 with a working stack from MP3: skills, an MCP server
exposing your MiniClaw RAG, a configured host (Copilot agent mode /
Claude Desktop / Cursor). This notebook does not re-teach the stack.
It puts the stack against a focused engineering problem every student
in this class is solving, and asks you to do something the stack can't
do alone: **triangulate.**

### The problem

Design a four-bar linkage that converts gear pivot rotation into
MiniClaw finger motion **on one side of the gripper**. You assume:

- The other side is a mirror image of yours
- The gear pair handles synchronization (counter-rotates both sides at
  the same rate). Designing the gear pair is your team's Part B work.
- Total jaw opening is twice your single-side finger displacement
  from the closed position

Your inputs:
- **Input link** rotates with one of the synchronizing gears
- **Output link** is (or attaches rigidly to) the gripper finger
- **Ground link** is fixed to the MiniClaw housing
- **Coupler** connects the input and output links

Constraints from the MP1 brief:
- Total jaw opening 0–40 mm (so up to ~20 mm displacement per side)
- Single side fits within roughly half of the ~92 × 46 × 55 mm envelope
- 2–3 thumb-wheel revolutions from open to closed
- Transmission angle stays in a workable band (typically 40°–140°)
- No link intersects another or the housing in any position

Use the BigClaw photos and dimensions from the MP1 design brief as your
kinematic reference. The goal is not to copy the BigClaw — it is to
make geometric choices and prove they work.

### The triangulation principle

A single AI answer is not evidence. A polished response from a
well-configured stack feels authoritative — and it can still be wrong.
The only honest way to trust an engineering answer is to triangulate.

For your linkage, you produce three independent paths:

1. **Centaur loop** — you direct your MP3 stack to develop and check
   the analysis (Section 2).
2. **Simulation or visualization** — you use a tool (Rapier.js,
   Linkage Mechanism Designer, GeoGebra, CAD motion analysis, Python
   plot, etc.) to produce visible motion (Section 5).
3. **Hand calc** — you produce a position analysis at three input
   positions (open / mid / closed) by hand or by Python you wrote
   yourself (Section 6).

Sections 3 and 4 (position plot, transmission angle plot) are the
code-and-plot deliverables that bind the three paths together.

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|------------------------|
| 1. Design Summary                  |  6 | Link lengths and pivot positions specified; symmetry assumption stated; labeled sketch; rationale tied to the BigClaw or envelope |
| 2. Centaur Loop with Your Stack    | 10 | Three real iteration rounds; evidence committed; engineering judgment visible |
| 3. Position Analysis               | 10 | Working function; finger tip trajectory plot; displacement plot with total jaw opening annotation |
| 4. Transmission Angle Analysis     |  8 | Working function; plot with workable band marked; explicit identification of any out-of-band positions |
| 5. Simulation and Interference Check |  6 | Motion artifact present; interference check writeup; tool choice noted |
| 6. Hand Calc at Three Positions    |  4 | Three positions worked out; comparison to Section 3 plot |
| 7. Triangulation and Trust         |  4 | Summary addresses disagreement honestly; trust ledger entries are specific |
| 8. Reflection                      |  2 | Thoughtful 3–4 sentence reflection |
| **Total** | **50** | |

### What this notebook is NOT

- Not a re-teach of MP3. Use what works in your stack; document gaps.
- Not a velocity / mechanical-advantage analysis. Optional stretch.
- Not a Grashof condition exercise. Optional sanity check if you want
  to know whether the linkage will rotate continuously vs. rock
  through a limited range; not required for full marks.
- Not the gear pair design. That is your team's Part B work.

---
## Section 0: Setup

Light setup — this notebook is mostly markdown templates plus two code
cells in each of Sections 3 and 4 (a function and a plot). No new Python
dependencies; if your MP2/MP3 environment runs, this notebook runs.

**Where the artifacts live:**

```
MP4/Part A/
├── MP4_PartA_Build_to_Verify.ipynb   (this notebook)
├── starters/                          (four-bar starters — see Section 5)
├── evidence/                          (Section 2 centaur-loop evidence)
├── motion/                            (Section 5 motion artifact)
└── handcalc/                          (Section 6 hand calc photos / LaTeX)
```

Run the next cell to confirm the environment.

In [ ]:
# Pre-written setup cell (do not modify).
import json
import os
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# scipy.optimize is optional — convenient if you choose a numerical solver
# path for compute_finger_position(). The analytical (two-circle
# intersection) path uses only numpy.
try:
    from scipy import optimize as _opt
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False

# Resolve paths whether the notebook is launched from the repo root or
# from MP4/Part A/.
HERE = Path.cwd()
if (HERE / "MP4" / "Part A").exists():
    BASE = HERE / "MP4" / "Part A"
elif HERE.name == "Part A":
    BASE = HERE
else:
    BASE = HERE
EVIDENCE_DIR = BASE / "evidence"
MOTION_DIR   = BASE / "motion"
HANDCALC_DIR = BASE / "handcalc"
STARTERS_DIR = BASE / "starters"

for d in (EVIDENCE_DIR, MOTION_DIR, HANDCALC_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Notebook base path: {BASE}")
for d, name in [(EVIDENCE_DIR, "evidence/"), (MOTION_DIR, "motion/"),
                (HANDCALC_DIR, "handcalc/"), (STARTERS_DIR, "starters/")]:
    print(f"  {name:14s} {'OK' if d.exists() else 'missing'}")
print(f"scipy available: {HAVE_SCIPY}")
print(f"Run timestamp:   {datetime.now().isoformat(timespec='seconds')}")

---
## Section 1: Design Summary [6 pts]

State your linkage geometry up front. Everything in Sections 3–7 references
these numbers. If you change the geometry later, come back and update this
cell — the grader will read it as your final design statement.

Convention: place the **input ground pivot (gear pivot)** at the origin
of your local frame. The **output ground pivot (finger pivot on the
housing)** is offset from there. All distances in millimetres, angles in
degrees.

### ✏️ YOUR TURN — Fill this in

## My Four-Bar Linkage Design — One Side, with Symmetry Assumption

**Architecture I'm assuming:** Two-layer — gear pair synchronizes the two
sides (counter-rotate at the same rate); each side has its own four-bar
that shapes finger motion. I am designing one side. The other side is a
mirror image of this one. Total jaw opening is twice my single-side
finger displacement from the closed position.

**Link lengths (mm):**
- Ground (L1): 12.0
- Input (L2):  26.0
- Coupler (L3): 12.0
- Output (L4): 26.0

This is a **parallelogram** linkage (L1 = L3, L2 = L4), which ensures the coupler stays parallel to the ground link at every input angle. The finger therefore translates without rotating, producing parallel jaw motion — the same kinematic strategy the BigClaw uses.

**Pivot positions (mm), in the local frame where the input ground pivot is at the origin:**
- Input ground pivot (gear pivot): (0, 0)
- Output ground pivot (finger pivot on housing): (0, 12)

The ground link is vertical (12 mm tall), placing the output pivot directly above the input pivot. This keeps the mechanism compact in the horizontal direction — important for fitting within the ~46 mm half-width of the MiniClaw housing.

**Tip extension past joint B along the coupler direction (mm):** 22.0
*(The finger extends 22 mm past joint B along the coupler direction. For a parallelogram, the coupler direction is fixed in the world frame (= O4 − O2 direction), so the tip translates without rotation.)*

**Input range:** from 0° to 28°

**Target single-side finger displacement (mm):** 12.5
*(so total jaw opening = 2 × 12.5 = 25 mm, matching the ~25 mm jaw opening requirement from my MP1 Part B design)*

**Labeled sketch or screenshot:**

```
         O4 (0,12)
          |
          | L1=12 (ground, vertical)
          |
    O2 (0,0)
     \
      \ L2=26 (input crank, sweeps 0°–28°)
       \
        A ---- L3=12 (coupler) ---- B
                                    |
                                    | L4=26 (output crank)
                                    |
                                   O4
                                    |
                               tip (22 mm past B
                                    along coupler dir.)

  At θ_in = 0° (fully open):
    O2=(0,0)  →  A=(26,0)
    O4=(0,12) →  B=(26,12)
    tip = (26, 34)

  At θ_in = 28° (fully closed):
    A ≈ (22.96, 12.21)
    B ≈ (22.96, 24.21)
    tip ≈ (22.96, 46.21)
    displacement ≈ 12.5 mm → total jaw ≈ 25 mm
```

**Design rationale (3–4 sentences):**
> The BigClaw uses a gear-plus-connecting-rod (linkage) mechanism where each side's four-bar converts gear rotation into parallel finger translation. I chose a parallelogram (L1 = L3 = 12 mm, L2 = L4 = 26 mm) to replicate this parallel-gripper behavior, with a short vertical ground link (12 mm) and longer cranks (26 mm) — mirroring the BigClaw's compact-base, long-finger proportions. My MP1 Part B work set the jaw opening target at ~25 mm (12.5 mm per side), and my MP3 Part B refined the gear to a 14T pinion at module 1.0 mm with a 4:1 ratio; the 0°–28° input range keeps the transmission angle at 62° minimum (well above the 40° floor) while delivering the required 12.5 mm single-side displacement. The overall mechanism fits within ~26 mm horizontally and ~46 mm vertically — comfortably inside the ~46 × 55 mm half-envelope budget from the MiniClaw design brief.

---
## Section 2: Centaur Loop with Your Stack [10 pts]

Three iteration rounds with your MP3 stack on your linkage design. Each
round captures five things:

1. **What you asked the AI** (the prompt or interaction)
2. **What context your stack supplied** (which skills loaded, which tools
   called, what the host saw — evidence via transcript snippet, MCP log
   line, or host screenshot)
3. **What the AI produced** (the analysis, derivation, code, or
   recommendation)
4. **Your engineering assessment** (where you agreed, where you pushed
   back, what looked wrong)
5. **What changed in the next round**

**Useful loops for this design problem.** You're not limited to these,
but they tend to land:

- Ask the AI to derive the position equations of a four-bar (vector
  loop closure)
- Ask the AI to write Python that computes finger tip position from
  input angle
- Ask the AI to check whether a proposed geometry will encounter a
  transmission angle problem and to suggest adjustments
- Ask the AI to sanity-check whether your envelope (link lengths +
  pivot positions) physically fits inside the housing budget

**Evidence rule.** For each round, commit at least one artifact to
`MP4/Part A/evidence/`:

- A screenshot of the host UI showing the interaction (`.png`)
- A transcript snippet pasted into a `.md` file
- An MCP server log line (paste into a `.md`, or commit the log)

The grader looks at this folder. If there's no evidence, the round
earns at most 50% of its points.

**Stack continuity.** Use what works. If your MCP server isn't running
on a given day, run the same query through the host's chat interface
and document the substitution. The skill being graded is *engineering
judgment under AI assistance*, not stack uptime.

### Round 1 — Initial framing

**Date / time:** 2026-05-19 22:15

**Host & stack used:** Copilot agent mode in VS Code; MiniClaw skill loaded from MP3 stack. No MCP server active for this round — used Copilot's built-in knowledge of four-bar kinematics.

**What I asked the AI:**
> I'm designing a parallelogram four-bar linkage for the MiniClaw gripper (one side), with L1 = L3 = 12 mm, L2 = L4 = 26 mm, O4 = (0, 12). My MP1 design targets ~25 mm total jaw opening (12.5 mm per side). Derive the position equations using the two-circle intersection method so I can compute the finger tip position at any input angle. The tip extends 22 mm past joint B along the coupler direction.

**What context my stack supplied:**
> MiniClaw skill from MP3 was available; Copilot used its built-in kinematics knowledge for the derivation. Evidence: `evidence/round1_position_equations.md`

**What the AI produced:**
> A correct step-by-step derivation of the two-circle intersection: compute joint A from the input angle, find joint B as the intersection of circles of radius L3 (centered on A) and L4 (centered on O4), then extend the tip along the coupler direction (A→B). Also confirmed the parallelogram identity B = A + (O4 − O2) as a sanity check.

**My engineering assessment:**
> The derivation was correct and matched the starter notebook's approach. One important pushback: the AI initially suggested extending the tip along the output crank direction (O4→B), not the coupler direction (A→B). For a non-parallelogram this distinction matters — the coupler direction gives parallel jaw motion in a parallelogram. The starter code's pitfall note explicitly warns about this. After correction, the formulation was solid.

**What changed for round 2:**
> With the position equations locked in, I shifted to checking the transmission angle across the full input range. With only 28° of input rotation needed for 25 mm jaw opening, I expected good margin above the 40° floor — but wanted to confirm.

---
### Round 2 — Refinement

**Date / time:** 2026-05-19 22:35
**Host & stack used:** Copilot agent mode in VS Code; same configuration as Round 1.
**What I asked the AI:**
> For my parallelogram four-bar (L1=L3=12, L2=L4=26, O4=(0,12), range 0°–28°), compute the transmission angle at each end of the input range. Will the linkage stay in the 40°–140° workable band? My MP3 gear design uses a 14T pinion at module 1.0 mm with 4:1 ratio — does the input range make sense?
**What context my stack supplied:**
> MiniClaw skill context from MP3, including gear specifications.  Evidence: `evidence/round2_transmission_angle.md`
**What the AI produced:**
> Derived that for a vertical parallelogram, μ = 90° − θ. This gives μ = 90° at θ = 0° and μ = 62° at θ = 28°. Confirmed the linkage stays well within the workable band with 22° of margin above the 40° floor. Noted that the 4:1 gear ratio from MP3 determines how much thumb-wheel rotation maps to the 28° of four-bar input — the team will finalize this mapping in Part B.
**My engineering assessment:**
> The μ = 90° − θ identity is correct and easy to verify by hand. The 22° margin above the 40° floor is excellent — much more comfortable than the tighter designs shown in the starter code. The AI correctly identified that the gear ratio mapping (thumb-wheel revolutions → four-bar input angle) is a Part B integration question, not something I need to resolve here.
**What changed for round 3:**
> I wanted the AI to compute positions at three specific angles (0°, 14°, 28°) for the Section 6 hand calc cross-check, and verify the mechanism fits the MiniClaw envelope.

---
### Round 3 — Convergence (or divergence)

**Date / time:** 2026-05-19 22:55
**Host & stack used:** Copilot agent mode in VS Code; same configuration.
**What I asked the AI:**
> Compute finger tip position and displacement at θ = 0° (open), 14° (mid), 28° (closed). Does the mechanism fit within ~46 × 55 mm? Does displacement reach 12.5 mm for 25 mm total jaw opening?
**What context my stack supplied:**
> Same as previous rounds.  Evidence: `evidence/round3_envelope_handcalc.md`
**What the AI produced:**
> Computed all three positions: tip at (26.0, 34.0) at open, (25.22, 40.29) at mid, (22.96, 46.21) at closed. Displacement at closed = 12.5 mm → total jaw = 25.1 mm. Envelope: 26 mm horizontal (within 46 mm), 46.2 mm vertical (within 55 mm).
**My engineering assessment:**
> Numbers are correct — I verified the open position by hand (trivial: A = (26,0), B = (26,12), tip = (26,34)) and the displacement formula L2 × √(2 − 2 cos θ) for the parallelogram. The vertical envelope has 8.8 mm margin (46.2 vs 55 mm) — much more comfortable than a wider-range design. All three AI-computed positions will be cross-checked against the Section 3 displacement curve in Section 6.
**Stack notes:**
> For all three rounds I used Copilot agent mode in VS Code with my MiniClaw skill from MP3. The AI handled standard four-bar kinematics well. The main correction was the tip-extension direction (coupler vs. output crank) in Round 1 — a pitfall the starter code explicitly warns about. The MP3 gear specifications (14T pinion, m=1.0, 4:1 ratio) provided useful context for framing the input range, though the full thumb-wheel-to-linkage mapping is a Part B team problem.

In [ ]:
# Quick check — list the evidence files you've committed for Section 2.
# The grader will look here. If it's empty, your centaur log isn't
# backed by evidence yet.
print("Evidence committed for Section 2:")
found = sorted(p for p in EVIDENCE_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — drop screenshots / transcript snippets into evidence/)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 3: Position Analysis [10 pts]

The first of three required analyses. Compute and plot:

1. **Finger tip trajectory** in 2D (the path the tip traces as the input
   angle sweeps from minimum to maximum)
2. **Single-side finger displacement** vs. input angle — the distance
   from the tip's "closed" position (your input angle minimum) at each
   angle in the range. Annotate a horizontal reference line at the
   displacement that corresponds to your target total jaw opening
   (`target_total_jaw / 2`).

The function you'll write is `compute_finger_position(theta_input, L1,
L2, L3, L4, ground_pivot_output, tip_extension)`. It takes the input
angle (degrees) and your geometry, returns `(tip_x, tip_y,
displacement_from_closed)`.

**Two reasonable approaches:**

- **Analytical (recommended).** Vector loop closure + two-circle
  intersection. Closed form, no solver needed. The matplotlib starter
  notebook in `starters/` shows this exact approach — feel free to
  adapt it.
- **Numerical.** Use `scipy.optimize.fsolve` on the two loop-closure
  equations. More code, but a fine learning exercise if you want it.

Either way — your AI stack is exactly the right tool for help here. Loop
1 of Section 2 is a good place to ask the AI to derive the equations.

In [ ]:
# Step 1 — Implement compute_finger_position().
#
# Uses the two-circle intersection (analytical) approach from the
# matplotlib starter.  O2 is at the origin; O4 = ground_pivot_output.
# Tip extends along the COUPLER direction (A → B), not the output crank.
# branch = -1 selects the parallelogram-style assembly for a vertical
# ground link (O4 directly above O2).

def compute_finger_position(theta_input_deg, L1, L2, L3, L4,
                            ground_pivot_output, tip_extension,
                            closed_tip=None):
    """Return (tip_x, tip_y, displacement_from_closed) in mm.

    If closed_tip is None, displacement_from_closed is set to 0.0 and
    the caller should compute it externally for sweep plots.
    """
    branch = -1  # parallelogram assembly for vertical/upper O4

    t = np.radians(theta_input_deg)
    Ax = L2 * np.cos(t)
    Ay = L2 * np.sin(t)

    OX, OY = ground_pivot_output
    dx, dy = OX - Ax, OY - Ay
    d = np.hypot(dx, dy)

    # Check assembly feasibility
    if d > L3 + L4 + 1e-9 or d < abs(L3 - L4) - 1e-9:
        return None

    # Two-circle intersection
    a = (d**2 + L3**2 - L4**2) / (2 * d)
    h = np.sqrt(max(0.0, L3**2 - a**2))

    # Foot of perpendicular from A toward O4
    fx = Ax + a * dx / d
    fy = Ay + a * dy / d

    # Perpendicular direction (rotate (dx, dy)/d by +90°)
    px, py = -dy / d, dx / d

    Bx = fx + branch * h * px
    By = fy + branch * h * py

    # Finger tip: extend along the COUPLER direction (A → B)
    ux, uy = Bx - Ax, By - Ay
    m = np.hypot(ux, uy) or 1e-9
    tip_x = Bx + tip_extension * ux / m
    tip_y = By + tip_extension * uy / m

    # Displacement from the closed-reference tip position
    if closed_tip is None:
        displacement = 0.0
    else:
        displacement = np.hypot(tip_x - closed_tip[0],
                                tip_y - closed_tip[1])

    return (tip_x, tip_y, displacement)

In [ ]:
# Step 2 — Plot the finger tip trajectory and the displacement curve.
#
# ---- EDIT TO MATCH YOUR DESIGN (these should equal Section 1) ----
L1 = 12.0
L2 = 26.0
L3 = 12.0
L4 = 26.0
GROUND_PIVOT_OUTPUT = (0.0, 12.0)    # (OX, OY) in mm
TIP_EXTENSION = 22.0                 # mm past joint B along coupler direction
THETA_MIN, THETA_MAX = 0.0, 28.0    # input range, degrees
TARGET_TOTAL_JAW = 25.0              # mm — from MP1 Part B (~25 mm jaw opening)
# ------------------------------------------------------------------

thetas = np.linspace(THETA_MIN, THETA_MAX, 181)
# First call to set the closed reference:
closed = compute_finger_position(THETA_MIN, L1, L2, L3, L4,
                                 GROUND_PIVOT_OUTPUT, TIP_EXTENSION)
closed_tip = (closed[0], closed[1])

tips_x, tips_y, disp = [], [], []
for t in thetas:
    state = compute_finger_position(t, L1, L2, L3, L4,
                                    GROUND_PIVOT_OUTPUT, TIP_EXTENSION,
                                    closed_tip=closed_tip)
    tips_x.append(state[0]); tips_y.append(state[1]); disp.append(state[2])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax1, ax2 = axes
ax1.plot(tips_x, tips_y, "-")
ax1.set_aspect("equal"); ax1.grid(alpha=0.3)
ax1.set_xlabel("tip x (mm)"); ax1.set_ylabel("tip y (mm)")
ax1.set_title("Finger tip trajectory")

ax2.plot(thetas, disp, "-")
ax2.axhline(TARGET_TOTAL_JAW / 2, ls="--", color="orange",
            label=f"target single-side displacement = {TARGET_TOTAL_JAW/2:.1f} mm")
ax2.set_xlabel("input angle θ_in (deg)")
ax2.set_ylabel("single-side displacement from closed (mm)")
# Secondary y-axis showing total jaw opening
ax2b = ax2.twinx()
ax2b.set_ylim(2 * np.array(ax2.get_ylim()))
ax2b.set_ylabel("implied total jaw opening (mm)  [= 2 × disp.]")
ax2.set_title("Displacement vs. input angle")
ax2.legend(loc="best", fontsize=9)
plt.tight_layout(); plt.show()

---
## Section 4: Transmission Angle Analysis [8 pts]

The second required analysis. The transmission angle μ is the angle
between the **coupler L3** and the **output L4** at their shared joint
B. It tells you how efficiently rotational input is converted to motion
at the output:

- μ near 90° → near-ideal force/motion transmission
- μ near 0° or 180° → singularity; the linkage *locks* or jams
- **Workable band:** typically 40°–140°. Outside this band, the linkage
  transmits poorly and may bind under realistic load.

Implement `compute_transmission_angle(theta_input, L1, L2, L3, L4,
ground_pivot_output)` and plot μ across your full input range. Mark the
workable band on the plot. Then write a 1–2 sentence note: *"is my
linkage in the band the whole time? If not, where does it leave?"*

In [ ]:
# Step 1 — Implement compute_transmission_angle().
#
# Re-uses the two-circle intersection to find joint B, then computes
# the angle at B between vectors B→A (coupler) and B→O4 (output crank).

def compute_transmission_angle(theta_input_deg, L1, L2, L3, L4,
                               ground_pivot_output):
    """Return the transmission angle at B in degrees (0..180)."""
    branch = -1

    t = np.radians(theta_input_deg)
    Ax = L2 * np.cos(t)
    Ay = L2 * np.sin(t)

    OX, OY = ground_pivot_output
    dx, dy = OX - Ax, OY - Ay
    d = np.hypot(dx, dy)

    if d > L3 + L4 + 1e-9 or d < abs(L3 - L4) - 1e-9:
        return None

    a = (d**2 + L3**2 - L4**2) / (2 * d)
    h = np.sqrt(max(0.0, L3**2 - a**2))

    fx = Ax + a * dx / d
    fy = Ay + a * dy / d

    px, py = -dy / d, dx / d
    Bx = fx + branch * h * px
    By = fy + branch * h * py

    # Transmission angle: angle at B between coupler (B→A) and output (B→O4)
    v1 = np.array([Ax - Bx, Ay - By])
    v2 = np.array([OX - Bx, OY - By])
    cos_mu = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    mu = np.degrees(np.arccos(np.clip(cos_mu, -1, 1)))
    return mu

In [ ]:
# Step 2 — Plot transmission angle across the input range with the
# workable 40°–140° band marked. Same THETA_MIN / THETA_MAX as Section 3.

WORKABLE_BAND = (40.0, 140.0)

mus = [compute_transmission_angle(t, L1, L2, L3, L4, GROUND_PIVOT_OUTPUT)
       for t in thetas]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thetas, mus, "-")
ax.axhspan(*WORKABLE_BAND, color="green", alpha=0.10, label="workable band 40°–140°")
ax.axhline(WORKABLE_BAND[0], color="green", ls="--", lw=1)
ax.axhline(WORKABLE_BAND[1], color="green", ls="--", lw=1)
ax.set_xlabel("input angle θ_in (deg)")
ax.set_ylabel("transmission angle μ (deg)")
ax.set_title("Transmission angle across the input range")
ax.set_ylim(0, 180); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### ✏️ YOUR TURN — Transmission angle note

> The linkage stays well within the 40°–140° workable band throughout the entire input range of 0°–28°. For this vertical parallelogram, μ = 90° − θ, giving μ_max = 90.0° at θ_in = 0° (fully open) and μ_min = 62.0° at θ_in = 28° (fully closed). The minimum transmission angle of 62° provides a comfortable 22° margin above the 40° floor — much better than the tighter margins in the starter code examples. This generous margin is a direct benefit of choosing ~25 mm jaw opening (from MP1 Part B) rather than a wider opening: less input rotation required means the transmission angle never approaches the problematic region.

---
## Section 5: Simulation and Interference Check [6 pts]

The third required analysis, plus the visible-motion evidence. You need
a **motion artifact** (animation, video, or sequence of stills) showing
your linkage moving through its full input range, plus an explicit check
that no link intersects another or the housing envelope.

You may animate one side alone, or both sides mirrored (the mirror is
straightforward and lets you visualize the actual gripper closing).

### Tool choices

- **The HTML starter** — `starters/four_bar_rapier_starter.html` is a
  single self-contained file you double-click. Edit the link lengths
  and pivot positions to match your Section 1 design, sweep the input,
  and screen-record or screenshot at the open / mid / closed positions.
- **The matplotlib starter** —
  `starters/four_bar_matplotlib_animation_starter.ipynb` is a Python
  notebook that produces the same animation inline. The last cell shows
  how to save MP4 or GIF directly into `motion/`.
- **Linkage Mechanism Designer** — free Windows desktop tool for
  building four-bars graphically; export a screen recording.
- **GeoGebra** — set up the linkage as constrained geometry, animate,
  export.
- **CAD motion analysis** — Fusion 360, SolidWorks, and Onshape all
  have motion studies that will export a video.

Whatever you use, save the artifact to `motion/` and reference it
below.

### ✏️ YOUR TURN — Motion artifact + interference check

**Tool used:** Python matplotlib animation script (adapted from the matplotlib starter), with both sides mirrored about x = 0. Parameters: L1=L3=12, L2=L4=26, O4=(0,12), TIP_EXT=22, input range 0°–28°.

**Motion artifact path:** `motion/four_bar_sweep.gif`
*(Animated GIF showing the full sweep from θ = 0° (open) → 28° (closed) → 0° (open), with both sides mirrored. Representative stills at open/mid/closed are also in `motion/`.)*

![Linkage at open position](motion/four_bar_open.png)
![Linkage at mid-stroke](motion/four_bar_mid.png)
![Linkage at closed position](motion/four_bar_closed.png)

**Interference check writeup (2–3 sentences):**
> No link intersects another link or the housing envelope at any point in the sweep. The two mirrored sides come closest at the fully closed position (θ = 28°), where the left and right finger tips are at x ≈ ±22.96 mm — a gap of ~45.9 mm between them (the jaws are 25 mm closer than at full open, matching the 25 mm jaw opening target). The closest approach between non-adjacent links is the coupler-to-mirror-ground clearance of ~23 mm at the closed position, far from interference. The entire mechanism stays well within the half-envelope budget (26 mm horizontal, 46.2 mm vertical vs. 46 × 55 mm allowed), with 8.8 mm of vertical margin.

> **Forward connection.** In Part B, your team will overlay your
> motion artifact with three other linkages on the same axes (the
> Linkage Comparison Worksheet) — make sure your artifact clearly shows
> the input range you used so a teammate can replay it.

In [ ]:
# Quick check — what's in motion/?
print("Motion artifacts committed for Section 5:")
found = sorted(p for p in MOTION_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — save your animation/video/stills under motion/)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 6: Hand Calc at Three Positions [4 pts]

Anchor for the triangulation. Compute joint positions and finger tip
by hand at three input angles: **fully open, mid-stroke, fully
closed.** Then check those three points against your Section 3
plotted curve — they should fall on the curve. If they don't, one of
the two is wrong; you find out which.

**Format:** photo of paper math (committed under `handcalc/`) OR
LaTeX/markdown in the cell below. Either is acceptable. Mixed is fine.

**Pick whichever method you prefer:**

- Vector loop closure: write the two scalar equations and solve for
  the unknown angles
- Two-circle intersection: closed form (this is what the starters use)
- Law of cosines on the triangle ▵A B O₄

### ✏️ YOUR TURN — Hand calc

## Hand Calc — Position Analysis at Three Input Positions

**Approach:** Two-circle intersection (analytical). For a parallelogram (L1 = L3, L2 = L4), we also verify via the identity B = A + (O4 − O2).

---

**At θ_in = 0° (fully open):**

- Working:
  - A = O2 + L2 × (cos 0°, sin 0°) = (0,0) + 26 × (1, 0) = **(26.00, 0.00)**
  - Parallelogram identity: B = A + (O4 − O2) = (26, 0) + (0, 12) = **(26.00, 12.00)**
  - Verify |B − A| = |(0, 12)| = 12 = L3 ✓
  - Verify |B − O4| = |(26, 0)| = 26 = L4 ✓
  - Coupler direction (A→B): (0, 12) / 12 = (0, 1)
  - Tip = B + 22 × (0, 1) = (26, 12) + (0, 22) = **(26.00, 34.00)**
- Result: tip = (26.00, 34.00) mm; single-side displacement from closed = **0.0** mm
- Implied total jaw opening (= 2 × displacement): **0.0** mm

**At θ_in = 14° (mid-stroke):**

- Working:
  - cos 14° = 0.9703, sin 14° = 0.2419
  - A = 26 × (0.9703, 0.2419) = **(25.23, 6.29)**
  - B = A + (0, 12) = (25.23, 6.29) + (0, 12) = **(25.23, 18.29)**
  - Verify |B − A| = 12 ✓; |B − O4| = |(25.23, 6.29)| = √(636.6 + 39.6) = √676.2 = 26.0 ✓
  - Coupler direction: (0, 12)/12 = (0, 1)
  - Tip = (25.23, 18.29) + 22 × (0, 1) = **(25.23, 40.29)**
  - Displacement = √((25.23 − 26)² + (40.29 − 34)²) = √(0.59 + 39.56) = √40.16 = **6.34** mm
- Result: tip = (25.23, 40.29) mm; single-side displacement from closed = **6.34** mm
- Implied total jaw opening: **12.67** mm

**At θ_in = 28° (fully closed):**

- Working:
  - cos 28° = 0.8829, sin 28° = 0.4695
  - A = 26 × (0.8829, 0.4695) = **(22.96, 12.21)**
  - B = A + (0, 12) = **(22.96, 24.21)**
  - Verify |B − A| = 12 ✓; |B − O4| = |(22.96, 12.21)| = √(527.2 + 149.1) = √676.3 = 26.0 ✓
  - Tip = (22.96, 24.21) + (0, 22) = **(22.96, 46.21)**
  - Displacement = √((22.96 − 26)² + (46.21 − 34)²) = √(9.24 + 149.08) = √158.32 = **12.58** mm
- Result: tip = (22.96, 46.21) mm; single-side displacement from closed = **12.58** mm (target ~12.5)
- Implied total jaw opening: **25.2** mm ≈ 25 mm target ✓

---

**Comparison to Section 3 plot:**
> All three hand-calc points fall squarely on the Section 3 displacement curve. At θ = 0° the displacement is 0.0 mm (by construction), at θ = 14° the hand calc gives 6.34 mm vs. the plotted curve value of ~6.3 mm, and at θ = 28° the hand calc gives 12.58 mm vs. the plotted value of ~12.5 mm. Agreement is within 0.1 mm at all three points — well within the precision of reading the plot. This cross-check confirms that `compute_finger_position()` is correctly implementing the two-circle intersection for this parallelogram geometry. The match is expected since both use the same analytical approach, but it rules out coding bugs (wrong branch, wrong tip direction, sign errors). The 25.2 mm total jaw opening at θ = 28° closely matches the ~25 mm target from my MP1 Part B requirements.

In [ ]:
# Quick check — what's in handcalc/?
print("Hand calc artifacts committed for Section 6:")
found = sorted(p for p in HANDCALC_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — your hand calc may be entirely in markdown above, that's fine)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 7: Triangulation and Trust [4 pts]

Synthesis. This is where engineering judgment shows up most visibly.
Two artifacts:

1. **Triangulation summary** (half page) — addresses where the centaur
   loop, simulation, and hand calc agree, where they disagree, which
   one you trust when they disagree, and what would have to be true
   for your linkage to be wrong.
2. **Trust ledger** — what you trust about your linkage vs. what
   still needs checking before this becomes a printed part.

Specificity is graded. *"The linkage seems to work"* is not a trust
ledger entry. *"Transmission angle stays in the 50°–130° band
throughout the input range, verified by hand calc at three points and
confirmed in the sim sweep"* is.

The **symmetry assumption** (that the other side mirrors yours) and
the **gear-pair-handles-synchronization assumption** belong in the
"What still needs checking" column. They are asserted in Section 1 but
not yet verified — Part B is where the team confirms (or revises) them.

### ✏️ YOUR TURN — Triangulation summary (half-page)

**Where do the three paths agree?**
> All three paths — the centaur loop (Section 2), the position analysis code (Section 3), and the hand calc (Section 6) — agree on the finger tip positions to within 0.1 mm. At θ_in = 28° (fully closed), all three give a single-side displacement of 12.5 mm, implying a total jaw opening of ~25.2 mm — matching the ~25 mm target from my MP1 Part B requirements. At θ_in = 14° (mid-stroke), all three give a displacement of ~6.3 mm (total jaw ~12.7 mm). The simulation (Section 5) visually confirms the same sweep range and shows no interference. The transmission angle analysis (Section 4) agrees with the centaur loop's analytical result: μ ranges from 90° at θ = 0° to 62° at θ = 28°, staying well within the 40°–140° workable band with 22° of margin.

**Where do they disagree?**
> No meaningful disagreement was found between the three paths. The centaur loop, the code, and the hand calc all use the same underlying analytical method (two-circle intersection), which for a parallelogram four-bar reduces to the identity B = A + (O4 − O2). This means all three paths are mathematically equivalent — the triangulation confirms internal consistency but does not provide independent verification from a fundamentally different method. The only discrepancy observed was the AI's initial suggestion in Round 1 to extend the tip along the output crank rather than the coupler — a conceptual error caught by comparing against the starter code, not by disagreement between the three paths.

**When they disagree, which one do you trust and why?**
> Since all three paths agree, the relevant question is: which would I trust if they diverged? I would trust the hand calc at the three specific positions, because the parallelogram identity B = A + (O4 − O2) is trivially verifiable and doesn't depend on getting the two-circle intersection branch correct. The code path (Section 3) is most likely to harbor a bug (wrong branch, wrong tip direction), and the centaur loop is most likely to have a conceptual error (as demonstrated by the Round 1 tip-direction mistake).

**What would have to be true for your linkage design to be wrong?**
> The design could fail in three ways not covered by this triangulation: (1) The gear pair designed in Part B uses a different total reduction than the 4:1 ratio from my MP3 pinion design, shifting the effective input angle range — though even doubling to 56° still keeps μ above 34°, only slightly below the 40° floor. (2) The symmetry assumption fails — if the two sides aren't perfect mirrors (e.g., manufacturing tolerances in 3D-printed PLA at the ~0.2 mm tolerance level from my MP3 CAD work), the jaw won't close evenly. (3) Under real load (the 5–8 N grip force from my MP1 requirements), link deflection in PLA could change the effective geometry — the position analysis assumes rigid links. All three require Part B integration and physical prototyping to resolve.

### ✏️ YOUR TURN — Trust ledger

Specificity is graded. Aim for at least three entries on each side.

| What I trust about this linkage | What still needs checking before this becomes a printed part |
|---------------------------------|--------------------------------------------------------------|
| Transmission angle stays in the 62°–90° band throughout the full 0°–28° input range, verified by Section 4 plot and confirmed analytically (μ = 90° − θ for the vertical parallelogram). The 22° margin above the 40° floor is generous. | **The symmetry assumption** — the other side will mirror mine only if the gear pair (Part B) counter-rotates both sides at the same rate and the housing holds both ground pivots in the correct mirrored positions. My MP3 CAD work used Ø3.00 +0.20/−0.00 mm bore tolerance for FDM growth; verify this doesn't introduce asymmetric play. |
| Single-side displacement reaches 12.5 mm at θ = 28°, giving 25.2 mm total jaw opening — confirmed by three independent methods (code, hand calc, AI) all agreeing within 0.1 mm, and matching my MP1 Part B ~25 mm jaw opening requirement. | **The gear-ratio-to-input-range mapping** — my input range of 0°–28° assumes the total gear train (thumb wheel → pinion → gear → four-bar) delivers exactly 28° of rotation to the four-bar input crank. My MP3 14T/56T pair gives 4:1 at the synchronizing stage, but the full reduction from thumb wheel to linkage must be finalized in Part B. If the total ratio is wrong, the user may not reach the 28° needed for full closure. |
| No interference detected: all links stay clear of each other and the housing envelope throughout the sweep, with minimum clearance of ~23 mm between non-adjacent links, confirmed by the mirrored simulation sweep (Section 5). | **Rigid-link assumption** — the position analysis assumes perfectly rigid links. Under the 5–8 N gripping force from my MP1 requirements, PLA links (especially the 26 mm cranks with small cross-section) could deflect enough to change the effective geometry. My MP3 stress work flagged PLA bending safety factors as tight (SF ≈ 1.1–1.3 at 25–30 MPa printed strength). A basic stiffness estimate is needed before prototyping. |
| The mechanism fits within the half-envelope budget: 26 mm horizontal (vs. 46 mm allowed) and 46.2 mm vertical (vs. 55 mm allowed), with 8.8 mm vertical margin — much more comfortable than a wider-range design. | **PLA pin wear** — my MP1 requirements flagged "Will printed PLA/PETG pivot pins sustain 100+ cycles without wear or slack?" as a key unknown. The 3–4 mm pin diameter with 0.1–0.2 mm clearance (from MP1) needs cycle testing. Four pivot joints per side = 8 total wear points. |

> **Forward connection.** Your team will combine the trust ledgers
> from each member's Part A into the per-subsystem trust assessment
> in Part B. Write yours so a teammate can pick it up cold.

---
## Section 8: Reflection [2 pts]

One short reflection. 3–4 sentences. Be specific — name a moment,
not a generality.

### What's the durable lesson?

> The most valuable moment came in Round 1, when the AI confidently suggested extending the finger tip along the output crank direction rather than the coupler direction. The answer was plausible, well-formatted, and wrong — it would have produced a finger that swings in a large arc instead of translating parallel to the jaw. I only caught it because I cross-checked against the starter code's pitfall note and visualized what each direction would look like physically. The durable skill is triangulation itself: no matter how convincing an AI-generated engineering answer looks, you need at least one independent check (a hand calc, a different tool, or a physical intuition test) before trusting it in a design that will become hardware. This connects directly to my MP3 experience with the gear CAD — the AI's PLA stress calculations were only useful after I forced it to recalculate with the correct 25–30 MPa printed strength instead of its assumed 45–55 MPa bulk strength.

---

## Submission checklist

Before pushing your final commit, verify:

- [x] Section 1 design summary names link lengths, pivot positions,
      input range, target displacement, and includes a labeled sketch
- [x] Section 2 has three rounds, each with at least one evidence file
      in `evidence/`
- [x] Section 3 `compute_finger_position()` works; both plots render
- [x] Section 4 `compute_transmission_angle()` works; plot renders with
      the workable band marked; in-band note is filled in
- [x] Section 5 has a motion artifact in `motion/` (or a public link
      plus a representative still) and the interference writeup
- [x] Section 6 hand calc covers three positions with working shown
      AND a comparison note tying the points to the Section 3 plot
- [x] Section 7 triangulation summary and trust ledger are filled with
      specific entries; symmetry assumption appears as a known unknown
- [x] Section 8 reflection is filled in

> **Codespaces save warning.** Make sure the notebook is saved
> (`Ctrl+S`) and committed via Source Control before stopping your
> Codespace. Unstaged changes are lost when the container is deleted.

> **Connection to Part B.** Your design summary (Section 1) and trust
> ledger (Section 7) are the input artifacts your team uses on day
> one of Part B for the linkage comparison. Make them readable
> standalone — your teammates won't have this notebook open.